# XGBoost Tennis Model Design

This notebook scaffolds the research and design for the Tennis Match Predictive Engine, as outlined in the deep research report. It is designed to be compatible with the `Artifact Contract` for the Rust runner, meaning we will define a strict feature schema and export the model to ONNX.

## Phase 1: Setup & Data Ingestion
- **Source:** Jeff Sackmann's ATP dataset (1968–2025)
- **Live Data:** Live tournament feeds, odds APIs

Prerequisites:
Make sure you have `xgboost`, `scikit-learn`, `onnxmltools` (or `skl2onnx`), and `polars` installed in your environment:
`pip install xgboost scikit-learn polars matplotlib onnxmltools`

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import matplotlib.pyplot as plt
import json
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

REPO_ROOT = Path.cwd().resolve().parent
DATA_DIR = REPO_ROOT / "data" / "tennis" / "tennis_atp" / "tennis_atp-master"
ARTIFACTS_DIR = REPO_ROOT / "artifacts" / "strategies" / "tennis_xgboost"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## Phase 2: Feature Engineering & Schema
According to the research, we need the following features:
1. **Elo Ratings** (Overall and surface-specific)
2. **Bookmaker Odds** (Implied probabilities)
3. **Head-to-Head (H2H)**
4. **Player Physicals** (Age, height differences)
5. **Momentum** (EWMA of recent win% or points won)

To ensure compatibility with the Rust runner, we define the schema precisely. Rust will expect feature vectors in this exact order.

In [ ]:
FEATURE_SCHEMA = {
  "schema_id": "tennis_xgboost_features",
  "schema_version": "1",
  "created_at": "2026-05-27T00:00:00Z",
  "features": [
    {"name": "elo_diff", "dtype": "float32", "nullable": False, "default": 0.0, "description": "P1 Elo - P2 Elo"},
    {"name": "age_diff", "dtype": "float32", "nullable": False, "default": 0.0, "description": "P1 Age - P2 Age"},
    {"name": "height_diff", "dtype": "float32", "nullable": False, "default": 0.0, "description": "P1 Height - P2 Height"},
    {"name": "p1_implied_prob", "dtype": "float32", "nullable": False, "default": 0.5, "description": "Implied win probability for P1 from odds"},
    {"name": "p2_implied_prob", "dtype": "float32", "nullable": False, "default": 0.5, "description": "Implied win probability for P2 from odds"}
    # Add H2H and momentum features here
  ],
  "target": {
    "name": "p1_win",
    "horizon_seconds": 86400  # 1 day roughly
  }
}

with open(ARTIFACTS_DIR / "feature_schema.json", "w") as f:
    json.dump(FEATURE_SCHEMA, f, indent=2)

def engineer_features(matches_df: pl.DataFrame) -> pl.DataFrame:
    """
    Stub for feature engineering. 
    """
    df = matches_df.with_columns([
        (pl.col("p1_elo") - pl.col("p2_elo")).cast(pl.Float32).alias("elo_diff"),
        (pl.col("p1_age") - pl.col("p2_age")).cast(pl.Float32).alias("age_diff"),
        (pl.col("p1_ht") - pl.col("p2_ht")).cast(pl.Float32).alias("height_diff"),
        (1 / pl.col("p1_odds")).cast(pl.Float32).alias("p1_implied_prob"),
        (1 / pl.col("p2_odds")).cast(pl.Float32).alias("p2_implied_prob")
    ])
    
    # IMPORTANT: Ensure the column order matches the schema exactly for Rust parity.
    ordered_cols = [f["name"] for f in FEATURE_SCHEMA["features"]]
    return df.select(ordered_cols + ["label"])

## Phase 3: Baseline XGBoost Model Training
Train a baseline model using `XGBClassifier` (scikit-learn API) as it makes ONNX export easier.

In [ ]:
def train_baseline_xgboost(X_train, y_train, X_val, y_val):
    print("Training XGBoost model...")
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric=["logloss", "auc", "error"],
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_estimators=500,
        early_stopping_rounds=50,
        random_state=42
    )
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50
    )
    return model

## Phase 4: KPI Evaluation & Export to ONNX
Exporting to ONNX is a requirement for the `Artifact Contract`.

In [ ]:
def evaluate_and_export_model(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    
    print("=== Model KPIs ===")
    print(f"Accuracy:   {acc:.4f} (Target: > 0.7500)")
    print(f"ROC AUC:    {auc:.4f} (Target: > 0.8500)")
    print(f"Log Loss:   {logloss:.4f} (Target: < 0.5000)")
    print(f"Brier Score:{brier:.4f} (Lower is better)")
    
    # Export to ONNX
    print("\nExporting model to ONNX for Rust runner...")
    num_features = len(FEATURE_SCHEMA["features"])
    initial_type = [('float_input', FloatTensorType([None, num_features]))]
    onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)
    
    onnx_path = ARTIFACTS_DIR / "model.onnx"
    onnxmltools.utils.save_model(onnx_model, str(onnx_path))
    print(f"Model saved to {onnx_path}")
    
    return y_prob

## Next Steps
1. **Download the Sackmann Data**: Fetch `atp_matches_*.csv` from the Jeff Sackmann GitHub repository and place them in `data/tennis/`.
2. **Implement Feature Builders**: Build out `add_h2h_features` and `add_momentum_features`.
3. **Run Experiments**: Connect the pipeline above, evaluate, and ensure the `.onnx` and `.json` artifacts are generated cleanly for parity testing.